# Фінальний проєкт. Інфекційні захворювання: схема `pandemic`, нормалізація, аналіз і власні функції

Дані: датасет `infectious_cases` з умови проєкту. У ньому кількість випадків інфекційних захворювань за країнами, регіонами й роками, 1920-2023. Копія CSV лежить у репозиторії: `data/infectious_cases.csv`.

| Таблиця | Що це |
|---|---|
| `pandemic.infectious_cases` | сирий імпорт CSV, усі колонки `TEXT` |
| `pandemic.entities` | довідник: унікальні пари `entity` + `code` з `id` |
| `pandemic.infectious_cases_normalized` | показники захворювань, де замість назви й коду стоїть `entity_id` |

PostgreSQL 16 запускається прямо в runtime через пакет `pixeltable-pgserver`. Оригінальний `pgserver` на поточній версії Python у Colab не встановлюється, а цей форк має той самий API.

Кожен крок має короткі блоки **Що робимо** і **Навіщо**, а після результату **Що вийшло**.

## Завдання 1. Завантаження даних

**Що робимо:** ставимо пакети, запускаємо PostgreSQL і перевіряємо підключення.

**Навіщо:** без сервера нема куди вантажити дані. `psycopg2` потрібен для `COPY FROM STDIN` через `copy_expert`, а `jupysql` дає клітинки `%%sql`.

In [1]:
%pip install -q "pixeltable-pgserver==0.4.0" psycopg2-binary sqlalchemy jupysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from urllib.request import urlretrieve

import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

# PostgreSQL усередині runtime; файли кластера лежать у /tmp/fp_pg, поки живе runtime
pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
# Драйвер psycopg2 явно: COPY робимо через його copy_expert
engine = create_engine(pg.get_uri(driver='psycopg2'), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.10 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [3]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)  # jupysql на Python 3.12+ сипле їх зі своїх docstring

%load_ext sql
%config SqlMagic.displaylimit = 50
%config SqlMagic.displaycon = False
%sql engine

**Що вийшло:** PostgreSQL 16.10 запустився всередині runtime, підключення через SQLAlchemy і `%%sql` працює.

### 1.1. CSV-файл

**Що робимо:** беремо `data/infectious_cases.csv`. Якщо файлу немає (наприклад, у Colab), качаємо його з Google Drive за посиланням з умови.

**Навіщо:** ментор запустить notebook і не шукатиме файл вручну. Копія в репозиторії знадобиться, якщо посилання колись перестане працювати.

In [4]:
DATA_DIR = 'data'
CSV_PATH = os.path.join(DATA_DIR, 'infectious_cases.csv')
# Пряме завантаження файлу за посиланням з умови проєкту
CSV_URL = ('https://drive.usercontent.google.com/download'
           '?id=1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl&export=download&confirm=t')

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(CSV_PATH):  # повторний запуск не качає файл ще раз
    urlretrieve(CSV_URL, CSV_PATH)
print(f'{CSV_PATH}: {os.path.getsize(CSV_PATH) / 1e6:.2f} MB')

with open(CSV_PATH, encoding='utf-8') as file:
    for _ in range(3):
        print(file.readline(), end='')

data/infectious_cases.csv: 0.54 MB
Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
Afghanistan,AFG,1980,,6160,0,,,,,,
Afghanistan,AFG,1981,,5859,0,,,,,,


**Що вийшло:** файл на місці (0.54 MB). У першому рядку заголовок з 12 колонок, далі йдуть рядки «країна, код, рік, показники». Порожнє поле означає, що даних за цей рік немає.

### 1.2. Схема `pandemic`

**Що робимо:** створюємо схему і робимо її схемою за замовчуванням.

**Навіщо:** усі таблиці й функції проєкту лежать в одній схемі. Після `SET search_path` можна писати `infectious_cases` замість `pandemic.infectious_cases`.

In [5]:
%%sql
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;

++
||
++
++

In [6]:
%%sql
SELECT current_schema() AS current_schema,
       current_setting('search_path') AS search_path;

1 rows affected.

current_schema,search_path
pandemic,pandemic


**Що вийшло:** поточна схема `pandemic`, і `search_path` теж `pandemic`. Наступні `%%sql`-клітинки працюють у тому самому з'єднанні, тож усі таблиці й функції далі створюються в цій схемі без префікса.

### 1.3. Таблиця `infectious_cases` та імпорт через COPY

**Що робимо:** створюємо `infectious_cases` з 12 колонками `TEXT` у порядку CSV і вантажимо файл через `COPY ... FROM STDIN`, як у ДЗ3.

**Навіщо:** у сирій таблиці все `TEXT`, тож імпорт не впаде ні на дробовому числі, ні на порожньому полі. Типи з'являться під час нормалізації. Імена колонок пишемо в нижньому регістрі: Postgres і так переводить у нього імена без лапок, тому `Number_rabies` у запитах стає `number_rabies`. У CSV-форматі COPY перетворює порожнє поле без лапок на `NULL`.

In [7]:
%%sql
DROP TABLE IF EXISTS infectious_cases;

-- Сирий імпорт CSV: 12 колонок у порядку файлу, усі TEXT
CREATE TABLE infectious_cases (
    entity               TEXT,
    code                 TEXT,
    year                 TEXT,
    number_yaws          TEXT,
    polio_cases          TEXT,
    cases_guinea_worm    TEXT,
    number_rabies        TEXT,
    number_malaria       TEXT,
    number_hiv           TEXT,
    number_tuberculosis  TEXT,
    number_smallpox      TEXT,
    number_cholera_cases TEXT
);

++
||
++
++

In [8]:
# COPY іде через окреме з'єднання без нашого search_path, тому ім'я таблиці зі схемою
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        cur.execute('TRUNCATE pandemic.infectious_cases')  # повторний запуск не дублює рядки
        with open(CSV_PATH, encoding='utf-8', newline='') as file:
            cur.copy_expert(
                'COPY pandemic.infectious_cases FROM STDIN WITH (FORMAT csv, HEADER true)',
                file,
            )
    raw_conn.commit()
finally:
    raw_conn.close()

In [9]:
%%sql
SELECT COUNT(*) AS loaded_rows
FROM infectious_cases;

1 rows affected.

loaded_rows
10521


**Що вийшло:** COPY без помилок завантажив усі 10 521 рядок файлу.

### 1.4. Огляд даних

**Що робимо:** дивимося на перші рядки, рахуємо entity, коди й роки, а потім перевіряємо кожну колонку з показниками: скільки в ній порожніх полів, скільки чисел і чи трапляється щось інше.

**Навіщо:** перед нормалізацією треба знати, що реально лежить у файлі. Від цього залежать типи колонок у завданні 2 і фільтр «лише числові значення» в завданні 5.

In [10]:
%%sql
SELECT *
FROM infectious_cases
ORDER BY entity, year
LIMIT 5;

5 rows affected.

entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
Afghanistan,AFG,1960,None,None,None,None,None,None,None,None,887
Afghanistan,AFG,1965,None,None,None,None,None,None,None,None,218
Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None


In [11]:
%%sql
SELECT COUNT(*)                                            AS n_rows,
       COUNT(DISTINCT entity)                              AS n_entities,
       (SELECT COUNT(*)
        FROM (SELECT DISTINCT entity, code FROM infectious_cases) AS pairs) AS n_entity_code_pairs,
       COUNT(DISTINCT entity) FILTER (WHERE code IS NULL)  AS n_entities_without_code,
       COUNT(*) FILTER (WHERE code IS NULL)                AS n_rows_without_code,
       MIN(year)                                           AS min_year,
       MAX(year)                                           AS max_year
FROM infectious_cases;

1 rows affected.

n_rows,n_entities,n_entity_code_pairs,n_entities_without_code,n_rows_without_code,min_year,max_year
10521,245,245,34,1128,1920,2023


In [12]:
%%sql
-- Чи однозначно code визначає entity і чи не повторюється пара (entity, year)
SELECT
    (SELECT COUNT(*)
     FROM (SELECT code FROM infectious_cases
           WHERE code IS NOT NULL
           GROUP BY code
           HAVING COUNT(DISTINCT entity) > 1) AS t) AS codes_with_several_entities,
    (SELECT COUNT(*)
     FROM (SELECT entity, year FROM infectious_cases
           GROUP BY entity, year
           HAVING COUNT(*) > 1) AS t)               AS duplicate_entity_year;

1 rows affected.

codes_with_several_entities,duplicate_entity_year
0,0


In [13]:
%%sql
-- Entity без коду (регіони, групи країн) і службові коди OWID_*
SELECT entity, code, COUNT(*) AS n_rows
FROM infectious_cases
WHERE code IS NULL OR code LIKE 'OWID%'
GROUP BY entity, code
ORDER BY code NULLS FIRST, entity;

37 rows affected.

entity,code,n_rows
Africa,None,52
African Region (WHO),None,30
Americas,None,41
Asia,None,73
East Asia & Pacific (WB),None,30
Eastern Mediterranean,None,41
Eastern Mediterranean Region (WHO),None,30
England,None,30
Europe,None,51
Europe & Central Asia (WB),None,30


In [14]:
%%sql
-- Профіль колонок з показниками: порожні поля, числа, інший текст
WITH v AS (
    SELECT c.ord, c.column_name, c.value
    FROM infectious_cases AS r
    CROSS JOIN LATERAL (VALUES
        (1, 'number_yaws',          r.number_yaws),
        (2, 'polio_cases',          r.polio_cases),
        (3, 'cases_guinea_worm',    r.cases_guinea_worm),
        (4, 'number_rabies',        r.number_rabies),
        (5, 'number_malaria',       r.number_malaria),
        (6, 'number_hiv',           r.number_hiv),
        (7, 'number_tuberculosis',  r.number_tuberculosis),
        (8, 'number_smallpox',      r.number_smallpox),
        (9, 'number_cholera_cases', r.number_cholera_cases)
    ) AS c(ord, column_name, value)
)
SELECT column_name,
       COUNT(*) FILTER (WHERE value IS NULL)                         AS n_empty,
       COUNT(*) FILTER (WHERE value ~ '^-?[0-9]+(\.[0-9]+)?$')       AS n_numeric,
       COUNT(*) FILTER (WHERE value ~ '^-?[0-9]+$')                  AS n_integer,
       COUNT(*) FILTER (WHERE value IS NOT NULL
                          AND value !~ '^-?[0-9]+(\.[0-9]+)?$')      AS n_other
FROM v
GROUP BY ord, column_name
ORDER BY ord;

9 rows affected.

column_name,n_empty,n_numeric,n_integer,n_other
number_yaws,10447,74,74,0
polio_cases,2911,7610,7610,0
cases_guinea_worm,2093,8428,8428,0
number_rabies,3741,6780,0,0
number_malaria,3741,6780,4108,0
number_hiv,3741,6780,51,0
number_tuberculosis,3741,6780,131,0
number_smallpox,10417,104,104,0
number_cholera_cases,7554,2967,2967,0


**Що вийшло:**

- 10 521 рядок, 245 entity і рівно 245 пар `entity` + `code`. Отже, у кожної entity один код, і жоден код не належить двом entity. Пара `(entity, year)` ніде не повторюється.
- У 34 entity коду немає зовсім (1 128 рядків): це континенти, регіони WHO і World Bank, групи країн за доходом, `G20`, `OECD Countries`, а також `England`, `Wales`, `Scotland`, `Northern Ireland` і `Serbia and Montenegro (former)`. Ще три entity мають службові коди `OWID_*`, серед них `World` (`OWID_WRL`).
- Роки від 1920 до 2023.
- У жодній колонці з показниками немає тексту, який не є числом (`n_other = 0`). Нечисловими в сирих даних бувають лише порожні поля, і COPY записав їх як `NULL`.
- `number_yaws`, `polio_cases`, `cases_guinea_worm`, `number_smallpox`, `number_cholera_cases` містять лише цілі числа, тож у нормалізованій таблиці вони отримають тип `INTEGER`. У `number_rabies`, `number_malaria`, `number_hiv`, `number_tuberculosis` є дроби (це модельні оцінки), тому для них буде `DOUBLE PRECISION`.
- `number_rabies` заповнений у 6 780 рядках, решта 3 741 порожні.

## Завдання 2. Нормалізація до 3НФ

**Що робимо:** розбиваємо `infectious_cases` на дві таблиці. Перша, `entities`, зберігає унікальні пари `entity` + `code` з `id`. Друга, `infectious_cases_normalized`, зберігає показники, де замість назви й коду стоїть `entity_id`.

**Навіщо:** у сирій таблиці назва й код повторюються в кожному рядку, у середньому 43 рази на entity (10 521 / 245). Код легко виправити в одному рядку й забути про решту. Після нормалізації назва записана один раз, а показники посилаються на неї через `id`. Заодно колонки отримують справжні типи замість `TEXT`.

In [15]:
%%sql
DROP TABLE IF EXISTS infectious_cases_normalized, entities;

-- Довідник: одна entity = один рядок; entity і code однозначно визначають одне одного
CREATE TABLE entities (
    id     INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity TEXT NOT NULL UNIQUE,
    code   TEXT UNIQUE  -- NULL у регіонів і груп без коду; UNIQUE допускає кілька NULL
);

-- Показники: один рядок = одна entity в один рік
CREATE TABLE infectious_cases_normalized (
    id                   INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity_id            INTEGER NOT NULL REFERENCES entities (id),
    year                 INTEGER NOT NULL,
    number_yaws          INTEGER,
    polio_cases          INTEGER,
    cases_guinea_worm    INTEGER,
    number_rabies        DOUBLE PRECISION,
    number_malaria       DOUBLE PRECISION,
    number_hiv           DOUBLE PRECISION,
    number_tuberculosis  DOUBLE PRECISION,
    number_smallpox      INTEGER,
    number_cholera_cases INTEGER,
    UNIQUE (entity_id, year)
);

++
||
++
++

**Що робимо:** заповнюємо довідник, потім таблицю показників. Текст перетворюємо на числа через `NULLIF(TRIM(...), '')::тип`.

**Навіщо:** таблицю показників з'єднуємо з довідником лише по `entity`. Якби з'єднували по парі `(entity, code)`, 1 128 рядків агрегатів без коду загубилися б: у них `code = NULL`, а порівняння `NULL = NULL` у SQL повертає NULL, тож такий рядок у JOIN не потрапляє.

In [16]:
%%sql
INSERT INTO entities (entity, code)
SELECT DISTINCT entity, code
FROM infectious_cases
ORDER BY entity;

-- JOIN лише по entity: у 34 entity code = NULL, а NULL = NULL не дає TRUE
INSERT INTO infectious_cases_normalized (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
)
SELECT e.id,
       r.year::INTEGER,
       NULLIF(TRIM(r.number_yaws), '')::INTEGER,
       NULLIF(TRIM(r.polio_cases), '')::INTEGER,
       NULLIF(TRIM(r.cases_guinea_worm), '')::INTEGER,
       NULLIF(TRIM(r.number_rabies), '')::DOUBLE PRECISION,
       NULLIF(TRIM(r.number_malaria), '')::DOUBLE PRECISION,
       NULLIF(TRIM(r.number_hiv), '')::DOUBLE PRECISION,
       NULLIF(TRIM(r.number_tuberculosis), '')::DOUBLE PRECISION,
       NULLIF(TRIM(r.number_smallpox), '')::INTEGER,
       NULLIF(TRIM(r.number_cholera_cases), '')::INTEGER
FROM infectious_cases AS r
JOIN entities AS e ON e.entity = r.entity
ORDER BY e.id, r.year::INTEGER;

245 rows affected.

10521 rows affected.

++
||
++
++

### 2.1. Перевірка нормалізації

**Що робимо:** порівнюємо кількість рядків, дивимося на вміст обох таблиць і збираємо рядки назад, щоб звірити їх із сирими даними.

**Навіщо:** нормалізація не повинна ні губити рядки, ні змінювати значення. Якщо `EXCEPT ALL` в обидва боки дає 0 рядків, з двох таблиць можна точно відновити вихідну.

In [17]:
%%sql
SELECT 'infectious_cases' AS table_name, COUNT(*) AS n_rows FROM infectious_cases
UNION ALL
SELECT 'entities', COUNT(*) FROM entities
UNION ALL
SELECT 'infectious_cases_normalized', COUNT(*) FROM infectious_cases_normalized;

3 rows affected.

table_name,n_rows
infectious_cases,10521
entities,245
infectious_cases_normalized,10521


In [18]:
%%sql
SELECT *
FROM entities
ORDER BY id
LIMIT 5;

5 rows affected.

id,entity,code
1,Afghanistan,AFG
2,Africa,None
3,African Region (WHO),None
4,Albania,ALB
5,Algeria,DZA


In [19]:
%%sql
SELECT *
FROM infectious_cases_normalized
ORDER BY id
LIMIT 5;

5 rows affected.

id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
1,1,1960,None,None,None,None,None,None,None,None,887
2,1,1965,None,None,None,None,None,None,None,None,218
3,1,1980,None,6160,0,None,None,None,None,None,None
4,1,1981,None,5859,0,None,None,None,None,None,None
5,1,1982,None,9730,0,None,None,None,None,None,None


In [20]:
%%sql
-- Декомпозиція без втрат: збираємо рядки назад і порівнюємо з raw у тих самих типах
WITH rebuilt AS (
    SELECT e.entity, e.code, n.year,
           n.number_yaws, n.polio_cases, n.cases_guinea_worm, n.number_rabies,
           n.number_malaria, n.number_hiv, n.number_tuberculosis,
           n.number_smallpox, n.number_cholera_cases
    FROM infectious_cases_normalized AS n
    JOIN entities AS e ON e.id = n.entity_id
),
raw_typed AS (
    SELECT entity,
           code,
           year::INTEGER                                           AS year,
           NULLIF(TRIM(number_yaws), '')::INTEGER                  AS number_yaws,
           NULLIF(TRIM(polio_cases), '')::INTEGER                  AS polio_cases,
           NULLIF(TRIM(cases_guinea_worm), '')::INTEGER            AS cases_guinea_worm,
           NULLIF(TRIM(number_rabies), '')::DOUBLE PRECISION       AS number_rabies,
           NULLIF(TRIM(number_malaria), '')::DOUBLE PRECISION      AS number_malaria,
           NULLIF(TRIM(number_hiv), '')::DOUBLE PRECISION          AS number_hiv,
           NULLIF(TRIM(number_tuberculosis), '')::DOUBLE PRECISION AS number_tuberculosis,
           NULLIF(TRIM(number_smallpox), '')::INTEGER              AS number_smallpox,
           NULLIF(TRIM(number_cholera_cases), '')::INTEGER         AS number_cholera_cases
    FROM infectious_cases
)
SELECT 'raw без rebuilt' AS check_name, COUNT(*) AS n_rows
FROM (SELECT * FROM raw_typed EXCEPT ALL SELECT * FROM rebuilt) AS missing
UNION ALL
SELECT 'rebuilt без raw', COUNT(*)
FROM (SELECT * FROM rebuilt EXCEPT ALL SELECT * FROM raw_typed) AS extra;

2 rows affected.

check_name,n_rows
raw без rebuilt,0
rebuilt без raw,0


**Що вийшло:** у `entities` 245 рядків, у `infectious_cases_normalized` 10 521, стільки ж, скільки в сирій таблиці. Зібрані назад рядки збігаються з сирими в обидва боки (0 і 0), тож нормалізація нічого не загубила й не змінила.

### 2.2. Чому це 3НФ

- **1НФ:** кожне поле містить одне значення, а кожен рядок має ключ (`id`, а також унікальну пару `(entity_id, year)`).
- **2НФ:** показники описують конкретну entity в конкретний рік, тобто залежать від усього ключа `(entity_id, year)`, а не від його частини. Назва й код залежать лише від entity, тому вони винесені в довідник.
- **3НФ:** у таблиці показників немає транзитивних залежностей. Раніше `code` залежав від `entity`, а `entity` від ключа рядка. Тепер `code` лежить у `entities` поруч з `entity`, а показники знають лише `entity_id`.

Можна піти ще далі й зробити «довгу» таблицю `(entity_id, year, disease, cases)`, де хвороба теж стає довідником. Умова просить саме дві таблиці, тож тут залишаємо широкий формат: одна колонка на хворобу.

## Завдання 3. Аналіз `number_rabies`

**Що робимо:** для кожної пари `entity` + `code` (через `id` довідника) рахуємо середнє, мінімум, максимум і суму `number_rabies`, сортуємо за середнім за спаданням і беремо 10 рядків.

**Навіщо:** так видно, де випадків сказу найбільше. `WHERE number_rabies IS NOT NULL` відкидає роки без даних. Чому він тут обов'язковий, показано одразу після результату. Для читабельності значення округлені до двох знаків, а сортування йде за неокругленим середнім.

In [21]:
%%sql
SELECT e.id,
       e.entity,
       e.code,
       ROUND(AVG(n.number_rabies)::NUMERIC, 2) AS avg_rabies,
       ROUND(MIN(n.number_rabies)::NUMERIC, 2) AS min_rabies,
       ROUND(MAX(n.number_rabies)::NUMERIC, 2) AS max_rabies,
       ROUND(SUM(n.number_rabies)::NUMERIC, 2) AS sum_rabies
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10;

10 rows affected.

id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
241,World,OWID_WRL,20192.37,14075.51,24744.66,605771.11
121,Lower Middle Income (WB),None,15193.96,10202.53,19182.80,455818.78
202,South Asia (WB),None,11729.89,7271.28,15361.88,351896.66
206,South-East Asia Region (WHO),None,11424.33,6806.01,15641.96,342729.81
77,G20,None,10189.05,6339.08,13164.88,305671.38
97,India,IND,8599.17,5425.87,11121.14,257975.20
209,Sub-Saharan Africa (WB),None,4748.27,4356.82,5017.98,142448.04
3,African Region (WHO),None,4649.70,4283.07,4917.78,139491.07
120,Low Income (WB),None,3568.11,3014.62,3873.53,107043.16
58,East Asia & Pacific (WB),None,3427.46,2009.94,4591.01,102823.83


**Що вийшло:** на першому місці `World`, у середньому 20 192 випадки на рік (від 14 076 до 24 745, сума за 30 років 605 771). Далі йдуть агрегати: Lower Middle Income (WB), South Asia (WB), South-East Asia Region (WHO), G20. Країна в топ-10 лише одна: India на 6-му місці, у середньому 8 599 випадків на рік. У 8 з 10 рядків `code` порожній, бо це регіони й групи країн без коду ISO, а `World` має службовий код `OWID_WRL`. Дані `number_rabies` є за 1990-2019 роки, тож сума складається з 30 річних значень.

### 3.1. Навіщо `WHERE number_rabies IS NOT NULL`

**Що робимо:** рахуємо entity без жодного значення `number_rabies` і запускаємо той самий запит без `WHERE`.

**Навіщо:** агрегати й так пропускають `NULL`. Але entity, у якої немає жодного значення, отримує `AVG = NULL`, а `ORDER BY ... DESC` у PostgreSQL за замовчуванням ставить `NULL` першими.

In [22]:
%%sql
SELECT COUNT(*)                              AS n_entities,
       COUNT(*) FILTER (WHERE n_values = 0) AS n_without_rabies
FROM (SELECT e.id, COUNT(n.number_rabies) AS n_values
      FROM entities AS e
      LEFT JOIN infectious_cases_normalized AS n ON n.entity_id = e.id
      GROUP BY e.id) AS per_entity;

1 rows affected.

n_entities,n_without_rabies
245,19


In [23]:
%%sql
-- Той самий запит без WHERE: першими йдуть entity без даних
SELECT e.entity, e.code, AVG(n.number_rabies) AS avg_rabies
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 3;

3 rows affected.

entity,code,avg_rabies
Reunion,REU,None
Oceania,None,None
South America,None,None


**Що вийшло:** у 19 з 245 entity немає жодного значення `number_rabies`. Без `WHERE` саме вони йдуть першими з `avg_rabies = NULL`, і топ-10 складався б з порожніх рядків. Можна було б написати `ORDER BY ... DESC NULLS LAST`, але фільтр ще й прибирає зайві групи з результату.

### 3.2. Додатково: лише країни

**Що робимо:** той самий запит лише для країн: відкидаємо entity без коду й зі службовими кодами `OWID_*`.

**Навіщо:** в основному топі більшість місць займають World і регіони, бо вони сумують десятки країн. Цей запит показує, які саме країни дають найбільше випадків. Основний запит вище зроблено рівно за умовою, а цей лише доповнює його.

In [24]:
%%sql
SELECT e.id,
       e.entity,
       e.code,
       ROUND(AVG(n.number_rabies)::NUMERIC, 2) AS avg_rabies,
       ROUND(MIN(n.number_rabies)::NUMERIC, 2) AS min_rabies,
       ROUND(MAX(n.number_rabies)::NUMERIC, 2) AS max_rabies,
       ROUND(SUM(n.number_rabies)::NUMERIC, 2) AS sum_rabies
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
  AND e.code IS NOT NULL
  AND e.code NOT LIKE 'OWID%'
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10;

10 rows affected.

id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
97,India,IND,8599.17,5425.87,11121.14,257975.20
163,Pakistan,PAK,1582.17,1177.14,1881.73,47465.09
151,Nigeria,NGA,1335.42,1073.95,1441.12,40062.46
42,China,CHN,1247.91,744.25,2075.59,37437.20
70,Ethiopia,ETH,1145.17,758.05,1323.40,34355.18
143,Myanmar,MMR,972.90,560.70,1449.83,29187.11
19,Bangladesh,BGD,785.71,180.40,1734.71,23571.26
146,Nepal,NPL,661.25,338.14,1041.63,19837.53
170,Philippines,PHL,618.69,358.06,898.07,18560.73
98,Indonesia,IDN,267.71,124.62,481.37,8031.44


**Що вийшло:** серед країн India далеко попереду (8 599 на рік у середньому), далі Pakistan (1 582), Nigeria (1 335), China (1 248) і Ethiopia (1 145).

## Завдання 4. Різниця в роках

**Що робимо:** для колонки `year` нормалізованої таблиці будуємо три атрибути вбудованими функціями. `MAKE_DATE(year, 1, 1)` повертає 1 січня цього року, `CURRENT_DATE` повертає сьогоднішню дату, а `EXTRACT(YEAR FROM AGE(...))` рахує кількість повних років між ними.

**Навіщо:** `AGE` рахує повні роки за календарем, так само як вік людини. Сьогоднішня дата щодня інша, тому атрибут рахуємо в запиті, а не зберігаємо в таблиці: збережене значення з часом застаріло б. Для прикладу беремо Україну за кілька років.

In [25]:
%%sql
SELECT e.entity,
       n.year,
       MAKE_DATE(n.year, 1, 1)                                                AS first_jan,
       CURRENT_DATE                                                           AS today,
       EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS years_diff
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE e.entity = 'Ukraine'
  AND n.year IN (1980, 1991, 2000, 2010, 2022)
ORDER BY n.year;

5 rows affected.

entity,year,first_jan,today,years_diff
Ukraine,1980,1980-01-01,2026-11-01,46
Ukraine,1991,1991-01-01,2026-11-01,35
Ukraine,2000,2000-01-01,2026-11-01,26
Ukraine,2010,2010-01-01,2026-11-01,16
Ukraine,2022,2022-01-01,2026-11-01,4


In [26]:
%%sql
-- Діапазон по всій таблиці
SELECT COUNT(*)  AS n_rows,
       MIN(year) AS min_year,
       MAX(year) AS max_year,
       MAX(EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER) AS max_years_diff,
       MIN(EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER) AS min_years_diff
FROM infectious_cases_normalized;

1 rows affected.

n_rows,min_year,max_year,max_years_diff,min_years_diff
10521,1920,2023,106,3


**Що вийшло:** на дату прогону (2026-11-01) від 1 січня 1980 минуло 46 повних років, а від 1 січня 2022 минуло 4. По всій таблиці різниця коливається від 3 років (2023) до 106 (1920). Початкова дата завжди 1 січня, тому результат дорівнює «поточний рік мінус year». `AGE` правильно рахує повні роки між будь-якими двома датами, а для 1 січня дає те саме число.

## Завдання 5. Власні функції

### 5.1. `years_since(year)`

**Що робимо:** загортаємо вираз із завдання 4 у функцію. На вході рік, на виході кількість повних років від 1 січня цього року до сьогодні.

**Навіщо:** щоб не переписувати `EXTRACT(YEAR FROM AGE(...))` у кожному запиті. Функція має позначку `STABLE`, а не `IMMUTABLE`, бо читає `CURRENT_DATE`: у межах одного запиту результат сталий, але завтра може бути іншим.

In [27]:
%%sql
CREATE OR REPLACE FUNCTION years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
$$;

++
||
++
++

In [28]:
%%sql
-- Приклад з умови: 1996 -> 1996-01-01 -> різниця в роках до сьогодні
SELECT 1996                   AS year,
       MAKE_DATE(1996, 1, 1)  AS first_jan,
       CURRENT_DATE           AS today,
       years_since(1996)      AS years_since;

1 rows affected.

year,first_jan,today,years_since
1996,1996-01-01,2026-11-01,30


In [29]:
%%sql
SELECT e.entity, n.year, years_since(n.year) AS years_since
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE e.entity = 'Ukraine'
  AND n.year IN (1980, 1991, 2000, 2010, 2022)
ORDER BY n.year;

5 rows affected.

entity,year,years_since
Ukraine,1980,46
Ukraine,1991,35
Ukraine,2000,26
Ukraine,2010,16
Ukraine,2022,4


In [30]:
%%sql
-- Звірка функції з вбудованим виразом із завдання 4 по всіх рядках
SELECT COUNT(*) AS n_rows,
       COUNT(*) FILTER (
           WHERE years_since(year)
                 <> EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER
       ) AS n_mismatches
FROM infectious_cases_normalized;

1 rows affected.

n_rows,n_mismatches
10521,0


**Що вийшло:** `years_since(1996)` повертає 30: стільки повних років минуло від 1996-01-01 до дати прогону. Для України функція дає ті самі числа, що й вираз у завданні 4, а звірка по всіх 10 521 рядках показує 0 розбіжностей.

### 5.2. `avg_cases_per_period(cases_per_year, divisor)`

**Що робимо:** функція ділить річну кількість захворювань на довільний дільник: 12 дає середнє на місяць, 4 на квартал, 2 на півріччя.

**Навіщо:** дільник приходить ззовні, тож може виявитися нулем. `NULLIF(p_divisor, 0)` перетворює нуль на `NULL`, і функція повертає `NULL` замість помилки ділення на нуль. Позначка `IMMUTABLE`, бо результат залежить лише від аргументів.

In [31]:
%%sql
CREATE OR REPLACE FUNCTION avg_cases_per_period(
    p_cases_per_year DOUBLE PRECISION,
    p_divisor        DOUBLE PRECISION
)
RETURNS DOUBLE PRECISION
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT p_cases_per_year / NULLIF(p_divisor, 0);
$$;

++
||
++
++

In [32]:
%%sql
-- Перевірка на простих числах: 1200 випадків на рік
SELECT avg_cases_per_period(1200, 12) AS per_month,
       avg_cases_per_period(1200, 4)  AS per_quarter,
       avg_cases_per_period(1200, 2)  AS per_half_year,
       avg_cases_per_period(1200, 0)  AS divisor_zero;

1 rows affected.

per_month,per_quarter,per_half_year,divisor_zero
100.0,300.0,600.0,None


**Що робимо:** застосовуємо функцію до `number_rabies` з дільниками 12, 4 і 2. Для прикладу беремо India за останні 10 років з даними.

**Навіщо:** умова просить брати лише записи, де кількість захворювань є числом. У нормалізованій таблиці `number_rabies` має тип `DOUBLE PRECISION`, тож кожне значення, яке не `NULL`, є числом. Профіль з розділу 1.4 показав, що нечисловими в сирих даних були тільки порожні поля, тому фільтр `IS NOT NULL` відкидає саме їх.

In [33]:
%%sql
SELECT e.entity,
       n.year,
       ROUND(n.number_rabies::NUMERIC, 2)                           AS rabies_per_year,
       ROUND(avg_cases_per_period(n.number_rabies, 12)::NUMERIC, 2) AS per_month,
       ROUND(avg_cases_per_period(n.number_rabies, 4)::NUMERIC, 2)  AS per_quarter,
       ROUND(avg_cases_per_period(n.number_rabies, 2)::NUMERIC, 2)  AS per_half_year
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
  AND e.entity = 'India'
ORDER BY n.year DESC
LIMIT 10;

10 rows affected.

entity,year,rabies_per_year,per_month,per_quarter,per_half_year
India,2019,5590.52,465.88,1397.63,2795.26
India,2018,5489.93,457.49,1372.48,2744.96
India,2017,5425.87,452.16,1356.47,2712.94
India,2016,5442.18,453.51,1360.54,2721.09
India,2015,5505.09,458.76,1376.27,2752.55
India,2014,5584.56,465.38,1396.14,2792.28
India,2013,5728.97,477.41,1432.24,2864.49
India,2012,5922.70,493.56,1480.68,2961.35
India,2011,6151.08,512.59,1537.77,3075.54
India,2010,6400.86,533.40,1600.21,3200.43


In [34]:
%%sql
-- Скільки записів підходять під фільтр і чи сходиться функція з діленням
SELECT COUNT(*) AS n_numeric_rows,
       COUNT(*) FILTER (WHERE avg_cases_per_period(number_rabies, 12) <> number_rabies / 12) AS n_mismatches_month,
       COUNT(*) FILTER (WHERE avg_cases_per_period(number_rabies, 4)  <> number_rabies / 4)  AS n_mismatches_quarter,
       COUNT(*) FILTER (WHERE avg_cases_per_period(number_rabies, 2)  <> number_rabies / 2)  AS n_mismatches_half_year
FROM infectious_cases_normalized
WHERE number_rabies IS NOT NULL;

1 rows affected.

n_numeric_rows,n_mismatches_month,n_mismatches_quarter,n_mismatches_half_year
6780,0,0,0


**Що вийшло:** на простих числах 1200 випадків на рік дають 100 на місяць, 300 на квартал і 600 на півріччя, а дільник 0 повертає `NULL`. Для India у 2019 році 5 590,52 випадку на рік дають 465,88 на місяць, 1 397,63 на квартал і 2 795,26 на півріччя. Фільтр пропустив 6 780 записів з числовим `number_rabies`, і на всіх функція збігається з прямим діленням на 12, 4 і 2.

## Підсумок

| Завдання | Що зроблено | Результат |
|---|---|---|
| 1 | схема `pandemic`, `SET search_path`, таблиця `infectious_cases` (12 колонок `TEXT`), імпорт через `COPY` | 10 521 рядок; нечислових значень, крім порожніх полів, немає |
| 2 | `entities` + `infectious_cases_normalized`, 3НФ | 245 і 10 521 рядків, `EXCEPT ALL` в обидва боки дає 0 |
| 3 | `AVG`, `MIN`, `MAX`, `SUM` для `number_rabies`, топ-10 за середнім | перший `World` (20 192 на рік), єдина країна в топі: India (8 599) |
| 4 | `MAKE_DATE`, `CURRENT_DATE`, `AGE` | від 3 до 106 повних років на дату прогону |
| 5 | `years_since`, `avg_cases_per_period` | 0 розбіжностей з вбудованим виразом; дільники 12, 4, 2 на 6 780 записах |